<a href="https://colab.research.google.com/github/kwamivi/vizuara-webinar-repo/blob/main/case_study_diffusionpolicyforrobotics_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# Case Study: Predicting the Next Word in Clinical Notes — An RNN-Powered Autocomplete Engine for Emergency Physicians
## Implementation Notebook

*Vizuara Case Study Series*
*Estimated time: 60–90 minutes*

## Context

Emergency physicians at MedScribe AI's partner hospitals spend nearly **44% of their shift time** on documentation — time stolen from the bedside. MedScribe's existing assistant offers only rigid phrase templates, and physicians have been disabling it at an accelerating rate. The ML team's mandate is clear: replace the static engine with a **sequential character-level autocomplete model** that reads the partial clinical note typed so far and predicts what comes next.

Why character-level? Clinical text is full of irregular tokens — medication names, abbreviations, vital-sign strings — that break word-level vocabularies. A character-level model has no "unknown token" problem, its final softmax layer is tiny (great for the 50 ms latency budget), and it handles mid-word suggestions naturally.

Why an RNN? Because the context matters. A physician who has typed *"68-year-old male, hypertensive, with diaphoresis and"* is heading somewhere very different from one who typed *"24-year-old female athlete with isolated."* A feedforward window model truncates or ignores that history. An RNN compresses the **entire preceding context** into a fixed-size hidden state and carries it forward — that is the core architectural insight we will build from scratch today.

In this notebook we will:
1. Build and explore a synthetic clinical-text corpus
2. Implement a character-level RNN **from scratch in PyTorch** (no `nn.RNN` shortcuts!)
3. Train it with Backpropagation Through Time (BPTT)
4. Visualise the training dynamics and loss landscape
5. Generate autocomplete suggestions and evaluate their quality

Let us get started.

## 3.1 Environment Setup and Imports

Before we write a single line of model code, let us make sure our environment is consistent and reproducible. We pin random seeds everywhere — PyTorch, NumPy, and Python's own `random` module — so that every student running this notebook gets the same results.

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import random
import math
import time
import os
from collections import Counter

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Device selection ──────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Running on: {DEVICE}")
print(f"🔦  PyTorch version: {torch.__version__}")

## 3.2 Data Acquisition — Building a Synthetic Clinical Corpus

MedScribe's real corpus lives behind a HIPAA data-sharing agreement that requires Safe Harbor de-identification before any ML pipeline can touch it. For this teaching notebook we will generate a **synthetic clinical corpus** that mirrors the statistical properties of real emergency-department notes: medical abbreviations, vital-sign strings, medication names, and the characteristic sentence patterns of ED documentation.

This is not a compromise — it is good engineering practice. Building the entire pipeline against a synthetic corpus first means we can iterate rapidly, share the notebook freely, and then swap in the real corpus by changing exactly one variable.

In [ ]:
# ── Synthetic clinical note templates ─────────────────────────────────────────
# We compose notes from parameterised templates drawn from realistic clinical
# language. Each template slot is filled from a domain vocabulary.

CHIEF_COMPLAINTS = [
    "chest pain radiating to the left arm",
    "shortness of breath at rest",
    "acute onset altered mental status",
    "right lower quadrant abdominal pain",
    "syncope with diaphoresis",
    "severe headache with photophobia",
    "left-sided weakness and facial droop",
    "palpitations with near-syncope",
    "fever and productive cough",
    "lower extremity swelling and erythema",
]

AGES      = [str(a) for a in range(22, 89)]
SEXES     = ["male", "female"]
HISTORIES = [
    "hypertension, type 2 diabetes, and hyperlipidemia",
    "coronary artery disease and prior CABG",
    "atrial fibrillation on anticoagulation",
    "COPD with home oxygen",
    "chronic kidney disease stage 3",
    "no significant past medical history",
    "congestive heart failure with EF 35%",
    "obstructive sleep apnea on CPAP",
]

VITALS_TEMPLATES = [
    "BP {sbp}/{dbp}, HR {hr}, RR {rr}, T {temp}C, SpO2 {spo2}% on room air",
    "T {temp}C, BP {sbp}/{dbp} mmHg, pulse {hr} bpm, RR {rr}, O2 sat {spo2}%",
]

EXAM_FINDINGS = [
    "Alert and oriented x3. Heart regular rate and rhythm. Lungs clear to auscultation bilaterally.",
    "Mild distress. Tachycardic. Decreased breath sounds at right base.",
    "Diaphoretic and pale. JVD present. Bilateral lower extremity pitting edema.",
    "Well-appearing, no acute distress. Abdomen soft, tender in RLQ with guarding.",
    "Confused, GCS 13. Pupils equal and reactive. No focal neurological deficits.",
]

ASSESSMENTS = [
    "Acute coronary syndrome, rule out NSTEMI. Start aspirin, heparin drip.",
    "Community-acquired pneumonia. Start IV ceftriaxone and azithromycin.",
    "Hypertensive urgency. Oral labetalol 200mg given. Repeat BP in 30 minutes.",
    "Pulmonary embolism, high pre-test probability. CT-PA ordered stat.",
    "Appendicitis, clinical diagnosis. Surgery consulted. NPO, IV fluids.",
    "Transient ischemic attack. Admit for stroke workup. Aspirin 325mg loaded.",
    "Cellulitis without abscess. IV vancomycin initiated. Wound care ordered.",
]

def random_vitals():
    return {
        "sbp": random.randint(90, 190),
        "dbp": random.randint(55, 110),
        "hr":  random.randint(48, 138),
        "rr":  random.randint(12, 28),
        "temp": round(random.uniform(36.1, 39.8), 1),
        "spo2": random.randint(88, 100),
    }

def generate_note():
    """Generate one synthetic ED clinical note."""
    age        = random.choice(AGES)
    sex        = random.choice(SEXES)
    complaint  = random.choice(CHIEF_COMPLAINTS)
    history    = random.choice(HISTORIES)
    vitals_tpl = random.choice(VITALS_TEMPLATES)
    vitals_str = vitals_tpl.format(**random_vitals())
    exam       = random.choice(EXAM_FINDINGS)
    assessment = random.choice(ASSESSMENTS)

    note = (
        f"CHIEF COMPLAINT: {complaint}.\n\n"
        f"HISTORY OF PRESENT ILLNESS: {age}-year-old {sex} with history of "
        f"{history} presenting with {complaint}. "
        f"Symptoms began acutely approximately {random.randint(1,48)} hours prior to arrival. "
        f"Patient denies recent travel, sick contacts, or prior similar episodes.\n\n"
        f"VITAL SIGNS: {vitals_str}.\n\n"
        f"PHYSICAL EXAMINATION: {exam}\n\n"
        f"ASSESSMENT AND PLAN: {assessment}\n"
    )
    return note

# Generate corpus
NUM_NOTES = 3000
print(f"Generating {NUM_NOTES} synthetic clinical notes...")
notes      = [generate_note() for _ in range(NUM_NOTES)]
full_corpus = "\n\n---\n\n".join(notes)

print(f"✅ Corpus generated")
print(f"   Total characters : {len(full_corpus):,}")
print(f"   Total notes      : {NUM_NOTES:,}")
print(f"\n── Sample note (first 600 chars) ──────────────────────────────")
print(full_corpus[:600])

Let us take a moment to examine the character-level statistics of our corpus. Understanding the data before touching the model is a first-principles habit we always want to cultivate.

In [ ]:
# ── Character-level statistics ────────────────────────────────────────────────
char_counts = Counter(full_corpus)
vocab       = sorted(char_counts.keys())
VOCAB_SIZE  = len(vocab)

print(f"Unique characters (vocabulary size V): {VOCAB_SIZE}")
print(f"Most common characters:")
for ch, cnt in char_counts.most_common(10):
    display = repr(ch)
    bar = "█" * (cnt // 5000)
    print(f"  {display:6s}  {cnt:>8,}  {bar}")

# ── Visualise character frequency distribution ────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
chars_sorted = [c for c, _ in char_counts.most_common()]
freqs_sorted = [char_counts[c] for c in chars_sorted]
ax.bar(range(len(chars_sorted)), freqs_sorted, color="#4A90D9", edgecolor="none")
ax.set_xticks(range(len(chars_sorted)))
ax.set_xticklabels([repr(c)[1:-1] for c in chars_sorted],
                    rotation=90, fontsize=7)
ax.set_xlabel("Character")
ax.set_ylabel("Frequency")
ax.set_title("Character Frequency Distribution — Synthetic Clinical Corpus")
ax.set_yscale("log")
plt.tight_layout()
plt.show()

## 3.3 Building the Character Vocabulary

Every ML model needs integers, not raw strings. We create two lookup tables that are the backbone of every text-processing pipeline:
- `char_to_idx`: maps each character to a unique integer index
- `idx_to_char`: the reverse mapping, used when we decode model predictions back to text

In [ ]:
# ── Vocabulary mappings ────────────────────────────────────────────────────────
char_to_idx = {ch: idx for idx, ch in enumerate(vocab)}
idx_to_char = {idx: ch  for idx, ch in enumerate(vocab)}

print(f"Vocabulary size V = {VOCAB_SIZE}")
print(f"\nFirst 20 mappings (char → index):")
for ch in vocab[:20]:
    print(f"  {repr(ch):6s} → {char_to_idx[ch]:3d}")

In [ ]:
# ── One-hot encoding utility ───────────────────────────────────────────────────
# Recall from Section 2: x_t ∈ {0,1}^V is a one-hot vector.
# With V ≈ 70–90, this is cheap — no embedding table needed.

def char_to_one_hot(char: str) -> torch.Tensor:
    """
    Convert a single character to its one-hot encoded vector.

    Args:
        char: A single character present in our vocabulary.

    Returns:
        A 1-D tensor of shape (VOCAB_SIZE,) with a 1 at char_to_idx[char]
        and 0 everywhere else.
    """
    vec = torch.zeros(VOCAB_SIZE)
    vec[char_to_idx[char]] = 1.0
    return vec

# Quick smoke test
sample_char = "A"
oh = char_to_one_hot(sample_char)
print(f"One-hot for '{sample_char}': shape={oh.shape}, "
      f"non-zero at index {oh.argmax().item()} "
      f"(expected {char_to_idx[sample_char]})")
print(f"Sum of vector: {oh.sum().item()} (should be 1.0)")

## 3.4 Data Preparation — Sequence Pairs for Training

An RNN language model is trained on **(input sequence, target sequence)** pairs where the target is the input shifted by one character. If the input is `"patient has"`, the target is `"atient has "`. At every step, the model sees the current character and must predict the next one.

Let us build a dataset of fixed-length chunks from our corpus. This is called **truncated BPTT**: we unroll the computation graph for `SEQ_LENGTH` steps at a time rather than the full document length.

In [ ]:
# ── Hyperparameters (data) ─────────────────────────────────────────────────────
SEQ_LENGTH  = 100   # characters per training sequence (BPTT window)
BATCH_SIZE  = 64    # sequences per mini-batch

# ── Encode the full corpus as integer indices ──────────────────────────────────
corpus_indices = [char_to_idx[ch] for ch in full_corpus if ch in char_to_idx]
corpus_tensor  = torch.tensor(corpus_indices, dtype=torch.long)

print(f"Corpus encoded: {len(corpus_tensor):,} integer tokens")
print(f"First 20 indices : {corpus_tensor[:20].tolist()}")
print(f"Decoded back     : {''.join(idx_to_char[i] for i in corpus_tensor[:20].tolist())!r}")

In [ ]:
# ── Build (input, target) sequence pairs ──────────────────────────────────────
def make_sequence_pairs(corpus: torch.Tensor,
                         seq_len: int) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Slice the corpus into non-overlapping (input, target) pairs for training.

    Each input  x[i] is a sequence of seq_len characters: corpus[i : i+seq_len]
    Each target y[i] is the same window shifted right by 1: corpus[i+1 : i+seq_len+1]

    Args:
        corpus:  1-D LongTensor of character indices.
        seq_len: Length of each training sequence.

    Returns:
        inputs:  LongTensor of shape (num_sequences, seq_len)
        targets: LongTensor of shape (num_sequences, seq_len)
    """
    n = (len(corpus) - 1) // seq_len
    inputs  = torch.stack([corpus[i * seq_len : i * seq_len + seq_len]
                           for i in range(n)])
    targets = torch.stack([corpus[i * seq_len + 1 : i * seq_len + seq_len + 1]
                           for i in range(n)])
    return inputs, targets

inputs, targets = make_sequence_pairs(corpus_tensor, SEQ_LENGTH)

print(f"Dataset shape  — inputs : {inputs.shape}")
print(f"                targets : {targets.shape}")
print(f"\nExample pair (first sequence):")
print(f"  Input  : {''.join(idx_to_char[i] for i in inputs[0].tolist())!r}")
print(f"  Target : {''.join(idx_to_char[i] for i in targets[0].tolist())!r}")
print(f"\n(Notice: target is input shifted left by one character ✅)")

Let us create a clean PyTorch `DataLoader` so that training code stays tidy.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, random_split

# ── Train / validation split ──────────────────────────────────────────────────
dataset    = TensorDataset(inputs, targets)
n_val      = int(0.1 * len(dataset))
n_train    = len(dataset) - n_val
train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                 generator=torch.Generator().manual_seed(SEED))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                           drop_last=True)

print(f"Training sequences   : {len(train_ds):,}")
print(f"Validation sequences : {len(val_ds):,}")
print(f"Train batches / epoch: {len(train_loader):,}")

## 3.5 Building the RNN — From First Principles

Now for the centrepiece of this notebook. We will implement the RNN's recurrence equation **manually** — no `nn.RNN`, no `nn.LSTM`, no magic. Just matrix multiplications, a $\tanh$, and a softmax.

Recall the two equations we derived in Section 2:

$$h_t = \tanh\!\bigl(W_{hh}\, h_{t-1} + W_{xh}\, x_t + b_h\bigr)$$

$$\hat{y}_t = \text{softmax}\!\bigl(W_{hy}\, h_t + b_y\bigr)$$

There are **three weight matrices** and **two bias vectors** — that is all an RNN is. Let us build them one layer at a time.

In [ ]:
# ── Model hyperparameters ─────────────────────────────────────────────────────
HIDDEN_SIZE  = 256   # H: dimension of the hidden state vector
LEARNING_RATE = 3e-3
NUM_EPOCHS    = 20

print("RNN parameter count estimate:")
params = {
    "W_xh (input → hidden)"  : VOCAB_SIZE  * HIDDEN_SIZE,
    "W_hh (hidden → hidden)" : HIDDEN_SIZE * HIDDEN_SIZE,
    "b_h  (hidden bias)"     : HIDDEN_SIZE,
    "W_hy (hidden → output)" : HIDDEN_SIZE * VOCAB_SIZE,
    "b_y  (output bias)"     : VOCAB_SIZE,
}
total = 0
for name, count in params.items():
    print(f"  {name}: {count:>12,}")
    total += count
print(f"  {'─'*40}")
print(f"  {'TOTAL':30s}: {total:>12,}")

In [ ]:
class VanillaRNN(nn.Module):
    """
    A character-level Vanilla RNN implemented from first principles.

    At every time step t the network:
      1. Combines the previous hidden state h_{t-1} with the current
         one-hot input x_t via two learned linear transformations.
      2. Squashes the result through tanh to produce the new hidden state h_t.
      3. Projects h_t through a final linear layer to produce logits over
         the vocabulary (pre-softmax scores for the next character).

    The softmax is NOT applied inside forward() — we rely on PyTorch's
    CrossEntropyLoss which fuses log-softmax + NLL for numerical stability.
    """

    def __init__(self, vocab_size: int, hidden_size: int):
        super().__init__()
        self.vocab_size  = vocab_size
        self.hidden_size = hidden_size

        # ── Learnable parameters ───────────────────────────────────────────
        # W_xh : maps input x_t  (vocab_size)  → hidden pre-activation
        self.W_xh = nn.Linear(vocab_size,  hidden_size, bias=False)

        # W_hh : maps hidden h_{t-1} (hidden_size) → hidden pre-activation
        # This is the "memory" connection — what makes an RNN recurrent.
        self.W_hh = nn.Linear(hidden_size, hidden_size, bias=False)

        # b_h : single shared bias for the hidden pre-activation
        self.b_h  = nn.Parameter(torch.zeros(hidden_size))

        # W_hy : maps hidden h_t → output logits over vocabulary
        self.W_hy = nn.Linear(hidden_size, vocab_size, bias=True)

        # ── Weight initialisation ──────────────────────────────────────────
        # Xavier uniform for input and output projections.
        # Orthogonal for W_hh — this is a well-known trick to prevent
        # gradient explosion/vanishing at initialisation.
        nn.init.xavier_uniform_(self.W_xh.weight)
        nn.init.orthogonal_(self.W_hh.weight)
        nn.init.xavier_uniform_(self.W_hy.weight)

    def init_hidden(self, batch_size: int) -> torch.Tensor:
        """Return a zero hidden state of shape (batch_size, hidden_size)."""
        return torch.zeros(batch_size, self.hidden_size)

    def forward(self,
                x_indices: torch.Tensor,
                h_prev:     torch.Tensor
                ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass for ONE time step.

        Args:
            x_indices : LongTensor of shape (batch_size,) — current char indices.
            h_prev    : FloatTensor of shape (batch_size, hidden_size).

        Returns:
            logits    : FloatTensor of shape (batch_size, vocab_size).
            h_t       : FloatTensor of shape (batch_size, hidden_size).
        """
        # Step 1 ── Convert integer indices to one-hot vectors
        # x_t shape: (batch_size, vocab_size)
        x_t = F.one_hot(x_indices, num_classes=self.vocab_size).float()

        # Step 2 ── Hidden state update
        # h_t = tanh(W_xh @ x_t  +  W_hh @ h_{t-1}  +  b_h)
        h_t = torch.tanh(self.W_xh(x_t) + self.W_hh(h_prev) + self.b_h)

        # Step 3 ── Output projection (raw logits, no softmax here)
        logits = self.W_hy(h_t)

        return logits, h_t

Let us verify the model's shapes before touching any data. This single sanity check has saved countless hours of debugging.

In [ ]:
# ✅ Shape verification
torch.manual_seed(SEED)
_model     = VanillaRNN(VOCAB_SIZE, HIDDEN_SIZE).to(DEVICE)
_batch     = 4
_x_dummy   = torch.randint(0, VOCAB_SIZE, (_batch,)).to(DEVICE)
_h_dummy   = _model.init_hidden(_batch).to(DEVICE)
_logits, _h_new = _model(_x_dummy, _h_dummy)

assert _logits.shape == (_batch, VOCAB_SIZE), \
    f"❌ Expected logits shape ({_batch}, {VOCAB_SIZE}), got {_logits.shape}"
assert _h_new.shape == (_batch, HIDDEN_SIZE), \
    f"❌ Expected hidden shape ({_batch}, {HIDDEN_SIZE}), got {_h_new.shape}"
assert _h_new.abs().max().item() <= 1.0, \
    "❌ tanh output should be in [-1, 1]"

print(f"✅ logits shape  : {_logits.shape}  (batch_size × vocab_size)")
print(f"✅ h_t shape     : {_h_new.shape}  (batch_size × hidden_size)")
print(f"✅ h_t range     : [{_h_new.min():.4f}, {_h_new.max():.4f}] — bounded by tanh")
print(f"\nTotal trainable parameters: {sum(p.numel() for p in _model.parameters()):,}")

## 3.6 The Training Loop — BPTT in Action

Training an RNN language model involves three things happening simultaneously:
1. **Rolling the RNN** through `SEQ_LENGTH` time steps, accumulating logits and loss at every step.
2. **Calling `loss.backward()`**, which PyTorch handles via the unrolled computation graph (this *is* BPTT — the graph stretches `SEQ_LENGTH` layers deep).
3. **Gradient clipping** to prevent the exploding-gradient pathology that the $\tanh$ helps with but does not fully eliminate.

Let us build this loop carefully.

In [ ]:
def run_sequence(model:     VanillaRNN,
                 x_batch:   torch.Tensor,
                 y_batch:   torch.Tensor,
                 h:         torch.Tensor,
                 criterion: nn.Module
                 ) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Roll the RNN across all SEQ_LENGTH time steps for a batch.

    Args:
        model     : The VanillaRNN instance.
        x_batch   : LongTensor of shape (batch_size, seq_len) — input indices.
        y_batch   : LongTensor of shape (batch_size, seq_len) — target indices.
        h         : FloatTensor of shape (batch_size, hidden_size) — initial state.
        criterion : Loss function (CrossEntropyLoss).

    Returns:
        total_loss : Scalar tensor — mean cross-entropy over all steps.
        h          : Detached hidden state for the next sequence chunk.
    """
    batch_size, seq_len = x_batch.shape
    total_loss = torch.tensor(0.0, device=x_batch.device)

    for t in range(seq_len):
        x_t    = x_batch[:, t]            # (batch_size,)
        y_t    = y_batch[:, t]            # (batch_size,)
        logits, h = model(x_t, h)         # logits: (batch_size, vocab_size)
        loss_t = criterion(logits, y_t)   # scalar
        total_loss = total_loss + loss_t

    # Average over time steps so the magnitude is independent of seq_len
    total_loss = total_loss / seq_len

    # Detach hidden state: we carry the *values* but not the gradient graph
    # across sequence boundaries. This is "truncated BPTT".
    h = h.detach()
    return total_loss, h

In [ ]:
def train_one_epoch(model:      VanillaRNN,
                    loader:     DataLoader,
                    optimiser:  torch.optim.Optimizer,
                    criterion:  nn.Module,
                    clip_norm:  float = 5.0
                    ) -> float:
    """Train for one full pass over the DataLoader. Returns mean loss."""
    model.train()
    total_loss = 0.0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        h = model.init_hidden(x_batch.size(0)).to(DEVICE)

        optimiser.zero_grad()
        loss, _ = run_sequence(model, x_batch, y_batch, h, criterion)
        loss.backward()

        # Gradient clipping — vital for RNN stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)

        optimiser.step()
        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model:     VanillaRNN,
             loader:    DataLoader,
             criterion: nn.Module
             ) -> float:
    """Evaluate on validation data. Returns mean loss."""
    model.eval()
    total_loss = 0.0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        h       = model.init_hidden(x_batch.size(0)).to(DEVICE)
        loss, _ = run_sequence(model, x_batch, y_batch, h, criterion)
        total_loss += loss.item()

    return total_loss / len(loader)

## 3.7 Training the Model

We are now ready to train. Let us instantiate the model and optimiser, then run the training loop while recording loss history for visualisation.

In [ ]:
# ── Instantiate model, loss, optimiser ────────────────────────────────────────
model     = VanillaRNN(VOCAB_SIZE, HIDDEN_SIZE).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Learning rate scheduler: halve LR if val loss plateaus for 3 epochs
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimiser, mode="min", factor=0.5, patience=3, verbose=True
)

print(f"Model          : VanillaRNN  (H={HIDDEN_SIZE})")
print(f"Parameters     : {sum(p.numel() for p in model.parameters()):,}")
print(f"Optimiser      : Adam  (lr={LEARNING_RATE})")
print(f"Loss           : CrossEntropyLoss")
print(f"Epochs         : {NUM_EPOCHS}")
print(f"Seq length     : {SEQ_LENGTH}")
print(f"Batch size     : {BATCH_SIZE}")

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
train_losses, val_losses = [], []
best_val_loss = float("inf")
start_time    = time.time()

print(f"\n{'Epoch':>6}  {'Train Loss':>12}  {'Val Loss':>10}  "
      f"{'Train PPL':>10}  {'Val PPL':>8}  {'Time':>6}")
print("─" * 65)

for epoch in range(1, NUM_EPOCHS + 1):
    t0         = time.time()
    train_loss = train_one_epoch(model, train_loader, optimiser, criterion)
    val_loss   = evaluate(model, val_loader, criterion)
    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_rnn.pt")
        marker = " ← best"
    else:
        marker = ""

    elapsed = time.time() - t0
    print(f"{epoch:>6}  {train_loss:>12.4f}  {val_loss:>10.4f}  "
          f"{math.exp(train_loss):>10.2f}  {math.exp(val_loss):>8.2f}  "
          f"{elapsed:>5.1f}s{marker}")

total_time = time.time() - start_time
print(f"\n✅ Training complete in {total_time/60:.1f} minutes")
print(f"   Best validation loss : {best_val_loss:.4f}")
print(f"   Best validation PPL  : {math.exp(best_val_loss):.2f}")

## 3.8 Visualising Training Dynamics

A loss curve is not decoration — it is a diagnostic instrument. Let us examine it carefully.

In [ ]:
# ── Loss and perplexity curves ────────────────────────────────────────────────
epochs_range = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: Cross-entropy loss ──────────────────────────────────────────────────
ax = axes[0]
ax.plot(epochs_range, train_losses, "o-", color="#4A90D9",
        label="Train", linewidth=2, markersize=5)
ax.plot(epochs_range, val_losses,   "s--", color="#E74C3C",
        label="Validation", linewidth=2, markersize=5)
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title("Training & Validation Loss")
ax.legend()
ax.grid(alpha=0.3)

# ── Right: Perplexity (exp of loss) ──────────────────────────────────────────
ax = axes[1]
ax.plot(epochs_range, [math.exp(l) for l in train_losses],
        "o-",  color="#4A90D9", label="Train PPL", linewidth=2, markersize=5)
ax.plot(epochs_range, [math.exp(l) for l in val_losses],
        "s--", color="#E74C3C", label="Val PPL",   linewidth=2, markersize=5)
ax.set_xlabel("Epoch")
ax.set_ylabel("Perplexity")
ax.set_title("Perplexity (lower is better)")
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle("VanillaRNN Training Dynamics — Clinical Autocomplete",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# ── Interpretation note ───────────────────────────────────────────────────────
gap = val_losses[-1] - train_losses[-1]
print(f"\n📊 Diagnostic summary:")
print(f"   Final train loss : {train_losses[-1]:.4f}  (PPL {math.exp(train_losses[-1]):.2f})")
print(f"   Final val loss   : {val_losses[-1]:.4f}  (PPL {math.exp(val_losses[-1]):.2f})")
print(f"   Train-val gap    : {gap:.4f} — {'mild overfitting' if gap > 0.3 else 'healthy generalisation'}")

### What Does Perplexity Mean in Practice?

Perplexity is the standard evaluation metric for language models. Intuitively, **perplexity $= e^{\text{loss}}$** tells us how many equally likely characters the model is uncertain between at each step. A perplexity of 2.0 means the model is as uncertain as if it were choosing randomly between 2 characters. Our character vocabulary has ~70–90 entries, so a random model would have perplexity ≈ 80. Getting well below that means the model has genuinely learned clinical language structure.

Let us also visualise the hidden state activations to build intuition for what the RNN is computing internally.

In [ ]:
# ── Hidden state activation visualisation ────────────────────────────────────
# Feed a short clinical sentence through the model and record hidden states.

model.eval()
sample_text = "CHIEF COMPLAINT: chest pain radiating to the left arm.\n\nHISTORY"
h_states    = []

with torch.no_grad():
    h = model.init_hidden(1).to(DEVICE)
    for ch in sample_text:
        if ch not in char_to_idx:
            continue
        x_idx = torch.tensor([char_to_idx[ch]], device=DEVICE)
        _, h  = model(x_idx, h)
        h_states.append(h[0, :64].cpu().numpy())   # record first 64 dims

h_matrix = np.array(h_states)   # (seq_len, 64)

fig, ax = plt.subplots(figsize=(16, 5))
im = ax.imshow(h_matrix.T, aspect="auto", cmap="RdBu_r",
               vmin=-1, vmax=1, interpolation="nearest")
ax.set_xlabel("Time step (character position)")
ax.set_ylabel("Hidden unit index (first 64 of 256)")
ax.set_title("Hidden State Activations Over a Clinical Sentence")

# Annotate every 5th character
tick_pos   = list(range(0, len(sample_text), 5))
tick_chars = [repr(sample_text[i])[1:-1] for i in tick_pos]
ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_chars, rotation=45, fontsize=8)

plt.colorbar(im, ax=ax, label="tanh activation")
plt.tight_layout()
plt.show()

## 3.9 Generating Autocomplete Suggestions

We now load our best checkpoint and implement temperature-scaled sampling and beam search — exactly the decoding strategy described in Section 2 of the case study. This is what the physician-facing autocomplete dropdown uses.

In [ ]:
# ── Load best checkpoint ───────────────────────────────────────────────────────
model.load_state_dict(torch.load("best_rnn.pt", map_location=DEVICE))
model.eval()
print("✅ Best checkpoint loaded")


def encode_context(text: str,
                   max_chars: int = 200) -> torch.Tensor:
    """
    Encode a text string as a list of valid character indices.
    Characters not in vocabulary are silently skipped.
    We truncate to max_chars to keep inference fast.

    Args:
        text      : Raw text typed by the physician.
        max_chars : Maximum context length to encode.

    Returns:
        1-D LongTensor of character indices.
    """
    indices = [char_to_idx[ch] for ch in text[-max_chars:] if ch in char_to_idx]
    return torch.tensor(indices, dtype=torch.long)


def warm_up_hidden(model:   VanillaRNN,
                   context: torch.Tensor) -> torch.Tensor:
    """
    Roll the RNN over the physician's existing text to build up a
    context-rich hidden state before we start generating.

    Args:
        model   : Trained VanillaRNN.
        context : 1-D LongTensor of character indices.

    Returns:
        h : FloatTensor of shape (1, hidden_size) — the context hidden state.
    """
    h = model.init_hidden(1).to(DEVICE)
    with torch.no_grad():
        for idx in context:
            x = idx.unsqueeze(0).to(DEVICE)
            _, h = model(x, h)
    return h

In [ ]:
def sample_next_char(logits:      torch.Tensor,
                     temperature: float = 0.7) -> int:
    """
    Sample the next character index from the model's output logits
    using temperature scaling.

    Temperature τ controls the sharpness of the distribution:
      τ < 1 → sharper (more deterministic, safer for clinical text)
      τ > 1 → flatter (more exploratory, higher risk of hallucination)
      τ = 1 → unmodified softmax

    Args:
        logits      : 1-D FloatTensor of shape (vocab_size,).
        temperature : Sampling temperature τ > 0.

    Returns:
        Sampled character index as an integer.
    """
    # Scale logits by temperature, then convert to probabilities
    scaled_probs = F.softmax(logits / temperature, dim=-1)
    # Draw one sample from the categorical distribution
    return torch.multinomial(scaled_probs, num_samples=1).item()


def generate_completion(model:       VanillaRNN,
                         context_str: str,
                         num_chars:   int   = 40,
                         temperature: float = 0.7
                         ) -> str:
    """
    Generate a single completion of `num_chars` characters
    by autoregressively sampling from the model.

    Args:
        model       : Trained VanillaRNN.
        context_str : Text the physician has already typed.
        num_chars   : How many characters to generate.
        temperature : Sampling temperature.

    Returns:
        Generated continuation string.
    """
    context_tensor = encode_context(context_str)
    h              = warm_up_hidden(model, context_tensor)

    generated = []
    # Seed the first generation step with the last character of context
    if len(context_tensor) > 0:
        last_idx = context_tensor[-1].unsqueeze(0).to(DEVICE)
    else:
        last_idx = torch.tensor([char_to_idx[" "]], device=DEVICE)

    with torch.no_grad():
        for _ in range(num_chars):
            logits, h = model(last_idx, h)
            next_idx  = sample_next_char(logits[0], temperature)
            generated.append(idx_to_char[next_idx])
            last_idx  = torch.tensor([next_idx], device=DEVICE)

    return "".join(generated)

Now let us implement **beam search** — the algorithm that produces the top-3 candidate completions shown in the physician dropdown. Beam search maintains a set of *beams* (partial sequences), each with a cumulative log-probability score, and at each step expands every beam by every possible next character, keeping only the top-K.

In [ ]:
def beam_search(model:       VanillaRNN,
                context_str: str,
                beam_width:  int   = 3,
                num_chars:   int   = 40,
                temperature: float = 0.7
                ) -> list[tuple[str, float]]:
    """
    Generate the top-`beam_width` completions using beam search.

    Each beam is a tuple of (generated_string, cumulative_log_prob, hidden_state).
    At every step we expand all beams, score the candidates, and keep the best
    `beam_width` continuations.

    Args:
        model       : Trained VanillaRNN.
        context_str : Text the physician has already typed.
        beam_width  : Number of beams (= number of dropdown suggestions).
        num_chars   : Length of each generated completion.
        temperature : Temperature for logit scaling.

    Returns:
        List of (completion_string, score) tuples, sorted best-first.
        Score is the mean per-character log-probability (higher is better).
    """
    context_tensor = encode_context(context_str)
    h_init         = warm_up_hidden(model, context_tensor)

    # Seed character
    if len(context_tensor) > 0:
        seed_idx = context_tensor[-1].item()
    else:
        seed_idx = char_to_idx[" "]

    # Each beam: (text_so_far, cumulative_log_prob, hidden_state)
    beams = [("", 0.0, h_init)]

    with torch.no_grad():
        for step in range(num_chars):
            candidates = []

            for text, score, h in beams:
                # Determine the last character index to feed in
                if step == 0:
                    last_idx = torch.tensor([seed_idx], device=DEVICE)
                else:
                    last_char = text[-1]
                    if last_char not in char_to_idx:
                        continue
                    last_idx = torch.tensor([char_to_idx[last_char]], device=DEVICE)

                logits, h_new = model(last_idx, h)
                log_probs     = F.log_softmax(logits[0] / temperature, dim=-1)

                # Expand: take the top beam_width next characters
                topk_log_probs, topk_indices = log_probs.topk(beam_width)

                for lp, idx in zip(topk_log_probs.tolist(),
                                    topk_indices.tolist()):
                    candidates.append((
                        text + idx_to_char[idx],
                        score + lp,
                        h_new.clone()
                    ))

            # Prune to top beam_width candidates by cumulative score
            candidates.sort(key=lambda x: x[1], reverse=True)
            beams = candidates[:beam_width]

    # Return (text, mean_log_prob) — normalise by length for fair comparison
    results = [(text, score / num_chars) for text, score, _ in beams]
    return results

Let us now test our autocomplete engine on several realistic clinical prompts — the kind of partial sentences a physician would actually type.

In [ ]:
# ── Autocomplete demonstration ────────────────────────────────────────────────
TEST_PROMPTS = [
    "CHIEF COMPLAINT: chest pain",
    "VITAL SIGNS: BP 142/88, HR 96, RR",
    "ASSESSMENT AND PLAN: Acute coronary",
    "The patient is a 72-year-old male with",
    "PHYSICAL EXAMINATION: Alert and oriented",
]

print("=" * 72)
print("  🏥  MedScribe AI — Character-Level Autocomplete Demonstration")
print("=" * 72)

for prompt in TEST_PROMPTS:
    completions = beam_search(model, prompt, beam_width=3,
                               num_chars=50, temperature=0.7)

    print(f"\n📝 Physician has typed:\n   \"{prompt}\"")
    print(f"\n   💡 Autocomplete suggestions (beam search, τ=0.7):")
    for rank, (completion, score) in enumerate(completions, 1):
        # Show only up to the first newline for readability
        clean = completion.split("\n")[0]
        print(f"   [{rank}]  ...{clean}   (score: {score:.3f})")
    print("   " + "─" * 65)

Let us also explore how **temperature** affects the suggestions, mirroring the calibration study referenced in the case study. This is a critical product decision: too deterministic and suggestions feel robotic; too random and they produce clinically implausible text.

In [ ]:
# ── Temperature sensitivity analysis ─────────────────────────────────────────
PROMPT       = "ASSESSMENT AND PLAN: The patient presents with"
TEMPERATURES = [0.3, 0.7, 1.0, 1.5]
N_CHARS      = 50

print(f"📝 Prompt: \"{PROMPT}\"\n")
print(f"{'Temperature':>14}  {'Generated completion'}")
print("─" * 72)

for tau in TEMPERATURES:
    completion = generate_completion(model, PROMPT,
                                      num_chars=N_CHARS,
                                      temperature=tau)
    clean = completion.replace("\n", "↵")[:55]
    print(f"  τ = {tau:.1f}       ...{clean}")

print("\n💬 Clinical interpretation:")
print("   τ = 0.3  → Very deterministic. May be repetitive but always 'safe'.")
print("   τ = 0.7  → Balanced. Recommended for physician-facing suggestions.")
print("   τ = 1.0  → Standard softmax. Noticeable variety, occasional oddity.")
print("   τ = 1.5  → High entropy. Suggestions may be implausible or incoherent.")

## 3.10 TODO: Implement Gradient Clipping Diagnostics

Now it is your turn. One of the most important hyperparameters in RNN training is the **gradient clipping threshold**. If we clip too aggressively, learning slows because updates are always truncated. If we clip too loosely, gradient explosions can destabilise training entirely.

Your task is to write a function that instruments the training loop to **record the gradient norm before clipping** at every batch, and then visualise how often clipping is actually triggered.

In [ ]:
# ── TODO 3.10: Gradient norm diagnostics ──────────────────────────────────────

def compute_grad_norm(model: VanillaRNN) -> float:
    """
    Compute the global L2 norm of all gradients in the model.

    The global L2 norm is defined as:
        ||∇||_2 = sqrt( sum over all parameters p of sum(grad(p)^2) )

    This is exactly the quantity that torch.nn.utils.clip_grad_norm_
    computes and then clips — we want to observe it *before* clipping.

    Hints:
        - Iterate over model.parameters()
        - Check that p.grad is not None before accessing it
        - Accumulate p.grad.data.norm(2) ** 2 for each parameter
        - Return the square root of the total

    Args:
        model: The VanillaRNN whose gradients we want to measure.

    Returns:
        The global gradient L2 norm as a Python float.
    """
    # ── YOUR CODE HERE ────────────────────────────────────────────────────────
    total_norm_sq = 0.0
    # TODO: iterate over model.parameters()
    # TODO: for each param with a gradient, accumulate squared L2 norm
    # TODO: return sqrt(total_norm_sq)
    raise NotImplementedError("Implement compute_grad_norm!")
    # ── END YOUR CODE ─────────────────────────────────────────────────────────


def train_one_epoch_with_diagnostics(model:      VanillaRNN,
                                      loader:     DataLoader,
                                      optimiser:  torch.optim.Optimizer,
                                      criterion:  nn.Module,
                                      clip_norm:  float = 5.0
                                      ) -> tuple[float, list[float]]:
    """
    Train for one epoch, recording gradient norms at every batch.

    This is identical to train_one_epoch(), with one addition:
    call compute_grad_norm() AFTER loss.backward() but BEFORE
    clip_grad_norm_(), and append the result to grad_norms.

    Args:
        model      : The VanillaRNN.
        loader     : Training DataLoader.
        optimiser  : Adam optimiser.
        criterion  : CrossEntropyLoss.
        clip_norm  : Clipping threshold.

    Returns:
        mean_loss  : Average training loss for this epoch.
        grad_norms : List of per-batch gradient norms (before clipping).

    Hints:
        - The structure is almost identical to train_one_epoch().
        - The only additions are: (1) call compute_grad_norm(model)
          after backward(), and (2) collect the value in a list.
    """
    model.train()
    total_loss = 0.0
    grad_norms = []

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        h       = model.init_hidden(x_batch.size(0)).to(DEVICE)

        optimiser.zero_grad()
        loss, _ = run_sequence(model, x_batch, y_batch, h, criterion)
        loss.backward()

        # ── YOUR CODE HERE ────────────────────────────────────────────────────
        # TODO: call compute_grad_norm(model) and append to grad_norms
        # ── END YOUR CODE ─────────────────────────────────────────────────────

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
        optimiser.step()
        total_loss += loss.item()

    return total_loss / len(loader), grad_norms

In [ ]:
# ✅ Verification — run after implementing compute_grad_norm
# We will do a single mini-batch forward-backward to test your implementation.

def _test_grad_norm():
    _test_model = VanillaRNN(VOCAB_SIZE, HIDDEN_SIZE).to(DEVICE)
    _test_opt   = torch.optim.Adam(_test_model.parameters(), lr=1e-3)
    _criterion  = nn.CrossEntropyLoss()

    _xb = torch.randint(0, VOCAB_SIZE, (4, SEQ_LENGTH)).to(DEVICE)
    _yb = torch.randint(0, VOCAB_SIZE, (4, SEQ_LENGTH)).to(DEVICE)
    _h  = _test_model.init_hidden(4).to(DEVICE)

    _test_opt.zero_grad()
    _loss, _ = run_sequence(_test_model, _xb, _yb, _h, _criterion)
    _loss.backward()

    norm = compute_grad_norm(_test_model)
    assert isinstance(norm, float), "❌ Return value should be a Python float"
    assert norm > 0.0,              "❌ Gradient norm should be positive after backward()"
    assert norm < 1e6,              "❌ Gradient norm seems unreasonably large — check your formula"

    # Cross-check against PyTorch's own implementation
    ref_norm = 0.0
    for p in _test_model.parameters():
        if p.grad is not None:
            ref_norm += p.grad.data.norm(2).item() ** 2
    ref_norm = ref_norm ** 0.5
    assert abs(norm - ref_norm) < 1e-4, \
        f"❌ Your norm ({norm:.4f}) differs from reference ({ref_norm:.4f})"

    print(f"✅ compute_grad_norm works correctly!")
    print(f"   Computed norm : {norm:.4f}")
    print(f"   Reference norm: {ref_norm:.4f}")
    print(f"   Absolute error: {abs(norm - ref_norm):.2e}")

_test_grad_norm()

Once you have implemented `compute_grad_norm`, run the diagnostic epoch and plot the gradient norm distribution. We want to see: how often does the norm exceed the clipping threshold, and what does the distribution look like?

In [ ]:
# ── TODO: Run diagnostic epoch and visualise ──────────────────────────────────
# Uncomment and run after implementing the functions above.

# diag_model  = VanillaRNN(VOCAB_SIZE, HIDDEN_SIZE).to(DEVICE)
# diag_opt    = torch.optim.Adam(diag_model.parameters(), lr=LEARNING_RATE)
# diag_crit   = nn.CrossEntropyLoss()
# CLIP_THRESH = 5.0
#
# _, grad_norms = train_one_epoch_with_diagnostics(
#     diag_model, train_loader, diag_opt, diag_crit, clip_norm=CLIP_THRESH
# )
#
# fig, axes = plt.subplots(1, 2, figsize=(14, 4))
#
# # Left: gradient norm over batches
# axes[0].plot(grad_norms, alpha=0.7, color="#4A90D9", linewidth=0.8)
# axes[0].axhline(CLIP_THRESH, color="#E74C3C", linestyle="--",
#                 label=f"Clip threshold = {CLIP_THRESH}")
# axes[0].set_xlabel("Batch")
# axes[0].set_ylabel("Gradient L2 norm (before clipping)")
# axes[0].set_title("Gradient Norm Over Training Batches")
# axes[0].legend()
# axes[0].grid(alpha=0.3)
#
# # Right: histogram of gradient norms
# axes[1].hist(grad_norms, bins=50, color="#4A90D9", edgecolor="white", alpha=0.85)
# axes[1].axvline(CLIP_THRESH, color="#E74C3C", linestyle="--",
#                 label=f"Clip threshold = {CLIP_THRESH}")
# pct_clipped = 100 * sum(n > CLIP_THRESH for n in grad_norms) / len(grad_norms)
# axes[1].set_title(f"Gradient Norm Distribution  ({pct_clipped:.1f}% batches clipped)")
# axes[1].set_xlabel("Gradient L2 norm")
# axes[1].set_ylabel("Frequency")
# axes[1].legend()
# axes[1].grid(alpha=0.3)
#
# plt.tight_layout()
# plt.show()
# print(f"Batches with gradient explosion (norm > {CLIP_THRESH}): "
#       f"{sum(n > CLIP_THRESH for n in grad_norms)} / {len(grad_norms)}")

## 3.11 TODO: Character-Level Prediction Accuracy

We have been evaluating the model with perplexity (a continuous metric derived from the loss). Now let us compute a more interpretable metric: **top-1 accuracy** — the fraction of time steps at which the model's single most confident prediction matches the actual next character.

In [ ]:
# ── TODO 3.11: Top-1 character prediction accuracy ────────────────────────────

@torch.no_grad()
def compute_accuracy(model:    VanillaRNN,
                     loader:   DataLoader,
                     n_batches: int = 20
                     ) -> float:
    """
    Compute the top-1 character prediction accuracy on a DataLoader.

    For each time step, the model produces a logit vector over the vocabulary.
    The predicted character is argmax(logits). Accuracy = fraction of steps
    where argmax(logits) == true next character.

    We only evaluate on `n_batches` batches to keep this fast.

    Hints:
        - Roll through the sequence exactly as in run_sequence().
        - At each step, call logits.argmax(dim=-1) to get predictions.
        - Compare predictions to y_batch[:, t] to get per-step correctness.
        - Accumulate correct predictions and total predictions.
        - Divide at the end.

    Args:
        model     : Trained VanillaRNN.
        loader    : A DataLoader (use val_loader).
        n_batches : How many batches to evaluate on.

    Returns:
        accuracy  : Float in [0, 1].
    """
    model.eval()
    total_correct = 0
    total_tokens  = 0

    for batch_idx, (x_batch, y_batch) in enumerate(loader):
        if batch_idx >= n_batches:
            break

        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        h       = model.init_hidden(x_batch.size(0)).to(DEVICE)

        # ── YOUR CODE HERE ────────────────────────────────────────────────────
        # TODO: iterate over time steps (for t in range(x_batch.shape[1]))
        # TODO: call model(x_batch[:, t], h) to get logits and updated h
        # TODO: compute predicted = logits.argmax(dim=-1)
        # TODO: accumulate correct = (predicted == y_batch[:, t]).sum().item()
        # TODO: accumulate total_tokens
        raise NotImplementedError("Implement compute_accuracy!")
        # ── END YOUR CODE ─────────────────────────────────────────────────────

    return total_correct / total_tokens

In [ ]:
# ✅ Verification
def _test_accuracy_function():
    """
    Sanity checks for compute_accuracy:
      1. Result is a valid probability in [0, 1].
      2. An untrained (random) model should have accuracy ≈ 1/VOCAB_SIZE.
      3. Our trained model should be significantly better than random.
    """
    # Check 1: return type and range
    acc_trained = compute_accuracy(model, val_loader, n_batches=10)
    assert isinstance(acc_trained, float), "❌ Return a Python float"
    assert 0.0 <= acc_trained <= 1.0,      "❌ Accuracy must be in [0, 1]"

    # Check 2: random baseline
    random_model = VanillaRNN(VOCAB_SIZE, HIDDEN_SIZE).to(DEVICE)
    acc_random   = compute_accuracy(random_model, val_loader, n_batches=10)
    random_expected = 1.0 / VOCAB_SIZE
    assert acc_random < 5 * random_expected, \
        f"❌ Random model accuracy ({acc_random:.4f}) seems too high"

    # Check 3: trained > random
    assert acc_trained > acc_random, \
        "❌ Trained model should outperform the random model"

    print(f"✅ compute_accuracy works correctly!")
    print(f"   Random baseline accuracy  : {acc_random:.4f}  "
          f"(expected ≈ {random_expected:.4f}  =  1/{VOCAB_SIZE})")
    print(f"   Trained model accuracy    : {acc_trained:.4f}")
    print(f"   Improvement over random   : {acc_trained / acc_random:.1f}×")

_test_accuracy_function()

## 3.12 Visualising What the Model Has Learned

Let us build one more diagnostic: a **confusion matrix over the most common characters**. This shows us which characters the model confuses with each other — a rich source of insight into where it has captured structure and where it has not.

In [ ]:
# ── Character confusion matrix ─────────────────────────────────────────────────
@torch.no_grad()
def build_confusion_matrix(model:      VanillaRNN,
                             loader:    DataLoader,
                             top_k:     int = 20,
                             n_batches: int = 15
                             ) -> tuple[np.ndarray, list[str]]:
    """
    Build a confusion matrix for the top_k most frequent characters.

    Returns:
        conf_matrix : (top_k, top_k) array — conf_matrix[true, pred]
        labels      : List of character strings corresponding to each row/col
    """
    model.eval()
    # Identify top_k most frequent characters
    topk_chars = [ch for ch, _ in char_counts.most_common(top_k)]
    topk_idx   = [char_to_idx[ch] for ch in topk_chars]
    idx_to_row  = {vidx: row for row, vidx in enumerate(topk_idx)}

    conf = np.zeros((top_k, top_k), dtype=np.int64)

    for batch_num, (x_batch, y_batch) in enumerate(loader):
        if batch_num >= n_batches:
            break
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        h       = model.init_hidden(x_batch.size(0)).to(DEVICE)

        for t in range(x_batch.shape[1]):
            logits, h = model(x_batch[:, t], h)
            preds     = logits.argmax(dim=-1)   # (batch_size,)
            trues     = y_batch[:, t]            # (batch_size,)

            for true_idx, pred_idx in zip(trues.tolist(), preds.tolist()):
                if true_idx in idx_to_row and pred_idx in idx_to_row:
                    conf[idx_to_row[true_idx], idx_to_row[pred_idx]] += 1

    labels = [repr(ch)[1:-1] for ch in topk_chars]
    return conf, labels

conf_matrix, conf_labels = build_confusion_matrix(model, val_loader)

# Normalise rows so each row sums to 1 (recall per class)
conf_norm = conf_matrix.astype(float)
row_sums  = conf_norm.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1   # avoid division by zero
conf_norm /= row_sums

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(conf_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(conf_labels)))
ax.set_yticks(range(len(conf_labels)))
ax.set_xticklabels(conf_labels, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(conf_labels, fontsize=9)
ax.set_xlabel("Predicted character")
ax.set_ylabel("True character")
ax.set_title("Character Prediction Confusion Matrix (row-normalised recall)\n"
             "Top-20 most frequent characters")
plt.colorbar(im, ax=ax, label="Recall probability")
plt.tight_layout()
plt.show()

## Summary — What We Built, and Why It Matters

Let us take a moment to trace the arc of everything we have implemented.

### The Technical Journey

We started from raw text — a synthetic corpus of emergency department clinical notes — and built every component of a character-level language model from scratch:

| Component | What it does | Where it lives |
|---|---|---|
| **Vocabulary** | Maps each of the ~70–90 printable characters to an integer index | `char_to_idx`, `idx_to_char` |
| **One-hot encoding** | Converts each character index to a sparse binary vector $x_t \in \{0,1\}^V$ | `char_to_one_hot` |
| **VanillaRNN** | Implements $h_t = \tanh(W_{hh}h_{t-1} + W_{xh}x_t + b_h)$ and $\hat{y}_t = W_{hy}h_t + b_y$ with no framework shortcuts | `VanillaRNN.forward` |
| **BPTT training** | Unrolls the graph over `SEQ_LENGTH` steps, clips gradients, updates weights | `run_sequence`, `train_one_epoch` |
| **Temperature sampling** | Scales logits by $1/\tau$ before softmax to control generation diversity | `sample_next_char` |
| **Beam search** | Maintains 3 candidate completions simultaneously, pruning at each step | `beam_search` |
| **Diagnostics** | Gradient norm tracking, perplexity curves, hidden state visualisation, confusion matrix | Sections 3.8–3.12 |

### The Business Connection

Every design choice traces back to MedScribe AI's constraints:

- **Character-level vocabulary** → handles irregular clinical tokens (medication names, abbreviations, vital signs) without an unknown-token problem, and keeps the softmax head tiny for the 50 ms latency budget.
- **RNN hidden state** → compresses arbitrary-length preceding context, so the model knows whether the physician is writing a chest-pain note or a cellulitis note before predicting the next word.
- **Truncated BPTT** → trains on CPU-affordable GPU instances (A10G) without running out of memory on full-document-length computation graphs.
- **Temperature τ = 0.7** → calibrated in the product's physician usability study: aggressive enough to be deterministic and clinically safe, flexible enough to avoid robotic repetition.
- **Beam width = 3** → matches the three-item dropdown the UI team designed based on physician eye-tracking research.
- **< 80 MB model binary** → the `VanillaRNN` with `H=256` totals well under this budget, enabling deployment on hospital-network on-premise proxy servers without GPU infrastructure.

### What Comes Next

This notebook is the proof-of-concept. The MedScribe ML team's roadmap from here includes:

1. **Replacing one-hot inputs with learned character embeddings** — a small dense lookup table that lets similar characters share representation.
2. **Upgrading to LSTM or GRU** — gated architectures that address the vanishing-gradient problem we observed in the hidden-state visualisation, enabling longer-range dependencies.
3. **Subword tokenisation** — moving from characters to BPE tokens to improve throughput at the same latency budget.
4. **ONNX export and quantisation** — converting the trained model to INT8 for the CPU inference containers, targeting < 10 ms per forward pass.
5. **Evaluation on real de-identified notes** — measuring perplexity and physician acceptance rate on the 180,000-note corpus from the academic medical centre partners.

The core principles you have practised here — building from the recurrence equation, understanding every tensor shape, diagnosing with gradient norms and activation maps, connecting architecture choices to deployment constraints — carry forward to every one of those steps.

Welcome to the frontier of clinical NLP. 🏥

In [ ]:
# ── Final model summary ────────────────────────────────────────────────────────
print("=" * 60)
print("  VanillaRNN Clinical Autocomplete — Training Summary")
print("=" * 60)
print(f"  Vocabulary size V  : {VOCAB_SIZE}")
print(f"  Hidden size H      : {HIDDEN_SIZE}")
print(f"  Parameters         : {sum(p.numel() for p in model.parameters()):,}")
print(f"  Sequence length    : {SEQ_LENGTH}")
print(f"  Training sequences : {len(train_ds):,}")
print(f"  Best val loss      : {best_val_loss:.4f}")
print(f"  Best val PPL       : {math.exp(best_val_loss):.2f}")
print(f"  Random baseline PPL: {VOCAB_SIZE:.2f}")
print(f"  PPL improvement    : {VOCAB_SIZE / math.exp(best_val_loss):.1f}×")
print("=" * 60)
print("\n  The model is ready. Let us go save some physician time. 🩺")